# SEMIR LiTS IG-JEPA v2: Self-supervised pre-training on protected graphs

**Goal**: Use IG-JEPA self-supervised pre-training on Mode C (intensity-protected) graphs,
then fine-tune for tumor classification. Compare with vanilla GINE baseline (val Dice 0.32).

**Pipeline**:
1. Build Mode C graphs (reuse cache from `semir_lits_twostage.ipynb`)
2. IG-JEPA pre-training (topology-aware masking + VICReg)
3. Fine-tune with MLP probe for tumor classification
4. Lift to voxels, evaluate Dice

In [ ]:
import os, sys, copy, time, json, re, random, gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GINEConv, BatchNorm
from torch.optim.lr_scheduler import CosineAnnealingLR
from scipy.ndimage import binary_dilation
import fastloops

DATA_ROOT = "/scratch/ud3d4/acm_data/Data"
RESULTS_DIR = "/home/ud3d4/Desktop/SWOG/results/semir_jepa_v2"
GRAPH_CACHE = "/dev/shm/semir_jepa_v2_graphs"
TWOSTAGE_CACHE = "/dev/shm/semir_twostage_graphs"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(GRAPH_CACHE, exist_ok=True)

HU_MIN, HU_MAX = -50, 250
PSI, ALPHA = 5, 25
OVERLAP_TH = 0.10
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

def pr(msg=""):
    print(msg, flush=True)

def section(title):
    pr(f"\n{'='*70}")
    pr(f"  {title}")
    pr(f"{'='*70}")

pr(f"Device: {DEVICE}")
if torch.cuda.is_available():
    pr(f"GPU: {torch.cuda.get_device_name(0)}")
    pr(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB")

## Step 1: Data loading + graph building (Mode C)

In [ ]:
# ---- Data helpers (same as twostage notebook) ----

def load_and_convert(vid):
    ct = np.load(os.path.join(DATA_ROOT, "ct", f"volume-{vid}.npy")).astype(np.float32)
    seg = np.load(os.path.join(DATA_ROOT, "seg", f"segmentation-{vid}.npy")).astype(np.int32)
    ct_u8 = np.clip(ct, HU_MIN, HU_MAX)
    ct_u8 = ((ct_u8 - HU_MIN) / (HU_MAX - HU_MIN) * 255).round().astype(np.uint8)
    ct_u8 = np.ascontiguousarray(ct_u8[..., np.newaxis])
    return ct, seg, ct_u8

def discover_volumes():
    ct_dir = os.path.join(DATA_ROOT, "ct")
    vids = []
    for f in sorted(os.listdir(ct_dir)):
        m = re.match(r"volume-(\d+)\.npy", f)
        if m:
            vid = int(m.group(1))
            seg_path = os.path.join(DATA_ROOT, "seg", f"segmentation-{vid}.npy")
            if os.path.exists(seg_path):
                seg = np.load(seg_path)
                if (seg == 2).sum() > 0:
                    vids.append(vid)
    return sorted(vids)

def bbox_from_mask(mask, margin=32):
    coords = np.argwhere(mask)
    z0, y0, x0 = coords.min(axis=0)
    z1, y1, x1 = coords.max(axis=0) + 1
    z0 = max(z0 - margin, 0); y0 = max(y0 - margin, 0); x0 = max(x0 - margin, 0)
    z1 = min(z1 + margin, mask.shape[0]); y1 = min(y1 + margin, mask.shape[1]); x1 = min(x1 + margin, mask.shape[2])
    return (slice(z0, z1), slice(y0, y1), slice(x0, x1))

def make_intensity_protected(ct_u8_crop, organ_crop):
    liver_vals = ct_u8_crop[..., 0][organ_crop]
    liver_mean = liver_vals.mean()
    liver_std = liver_vals.std()
    candidate = organ_crop & (ct_u8_crop[..., 0] < liver_mean - 0.5 * liver_std)
    protected = binary_dilation(candidate, iterations=2)
    return protected.astype(np.uint8)

# ---- Feature extraction (from v7) ----

def _layout(C):
    return dict(area=0, s=[1, 2, 3],
                cov=[(4, 0, 0), (5, 1, 1), (6, 2, 2), (7, 0, 1), (8, 0, 2), (9, 1, 2)],
                chan0=10, boundary=10 + C + 6, D=3)

def node_invariants(node_feats, C=1, eps=1e-6):
    f = node_feats.astype(np.float64)
    L = _layout(C); D = L["D"]; N = f.shape[0]
    V = f[:, L["area"]]; Vsafe = np.maximum(V, 1.0)
    mean_coord = np.stack([f[:, c] for c in L["s"]], axis=1) / Vsafe[:, None]
    cov = np.zeros((N, D, D))
    for col, i, j in L["cov"]:
        cij = f[:, col] / Vsafe - mean_coord[:, i] * mean_coord[:, j]
        cov[:, i, j] = cij; cov[:, j, i] = cij
    w = np.linalg.eigvalsh(cov); w = np.clip(w, 0.0, None)
    _, vec = np.linalg.eigh(cov); principal = vec[..., -1]
    trace = w.sum(axis=1); degenerate = trace < eps
    denom = w[:, 2] + eps
    shape = np.stack([(w[:, 2] - w[:, 1]) / denom,
                      (w[:, 1] - w[:, 0]) / denom,
                      w[:, 0] / denom], axis=1)
    shape[degenerate] = 0.0
    line_like = np.where(degenerate, 0.0, shape[:, 0])
    chan = f[:, L["chan0"]:L["chan0"] + C] / Vsafe[:, None] / 255.0
    compactness = f[:, L["boundary"]] / np.power(Vsafe, (D - 1.0) / D)
    elongation = np.where(w[:, 0] > eps, w[:, 2] / (w[:, 0] + eps), 1.0)
    elongation = np.clip(elongation, 1.0, 100.0)
    return dict(V=V, surface=f[:, L["boundary"]], centroid=mean_coord,
                eig=w, shape=shape, line_like=line_like, principal=principal,
                chan=chan, compactness=compactness, elongation=elongation)

def edge_invariants(node_feats, edge_index, edge_feats, C=1, eps=1e-6):
    inv = node_invariants(node_feats, C, eps)
    a = edge_index[0].astype(np.int64); b = edge_index[1].astype(np.int64)
    ef = edge_feats.astype(np.float64); blsafe = np.maximum(ef[:, 0], 1.0)
    size_contrast = np.abs(inv["V"][a] - inv["V"][b]) / (inv["V"][a] + inv["V"][b] + eps)
    bfrac_a = ef[:, 0] / (inv["surface"][a] + eps)
    bfrac_b = ef[:, 0] / (inv["surface"][b] + eps)
    mean_contrast = np.abs(inv["chan"][a] - inv["chan"][b])
    shape_dissim = np.abs(inv["shape"][a] - inv["shape"][b])
    axis_align = (np.abs(np.sum(inv["principal"][a] * inv["principal"][b], axis=1))
                  * np.minimum(inv["line_like"][a], inv["line_like"][b]))
    bcontrast = (ef[:, 1] / blsafe) / 255.0
    cut_frac = ef[:, 3] / blsafe
    cols = [size_contrast[:, None], bfrac_a[:, None], bfrac_b[:, None],
            mean_contrast if mean_contrast.ndim > 1 else mean_contrast[:, None],
            shape_dissim, axis_align[:, None], bcontrast[:, None], cut_frac[:, None]]
    return np.concatenate(cols, axis=1).astype(np.float32)

def compute_intensity_std(labels_np, ct_u8):
    flat = labels_np.ravel(); valid = flat >= 0
    if not valid.any(): return np.array([], dtype=np.float32)
    max_id = int(flat[valid].max())
    vals = ct_u8[..., 0].ravel().astype(np.float64) / 255.0
    counts = np.bincount(flat[valid], minlength=max_id + 1).astype(np.float64)
    sums = np.bincount(flat[valid], weights=vals[valid], minlength=max_id + 1)
    sq_sums = np.bincount(flat[valid], weights=vals[valid] ** 2, minlength=max_id + 1)
    mean = sums / np.maximum(counts, 1.0)
    var = sq_sums / np.maximum(counts, 1.0) - mean ** 2
    return np.sqrt(np.maximum(var, 0.0)).astype(np.float32)

def build_pyg_graph(raw_nf, raw_ei, raw_ef, labels_np, seg, ct_u8, overlap_th, C=1):
    n_sn = raw_nf.shape[0]
    inv = node_invariants(raw_nf, C)
    int_std = compute_intensity_std(labels_np, ct_u8)
    if len(int_std) < n_sn: int_std = np.pad(int_std, (0, n_sn - len(int_std)))
    int_std = int_std[:n_sn]
    principal = inv["principal"].astype(np.float32)
    if len(principal):
        max_comp = np.argmax(np.abs(principal), axis=1)
        signs = np.sign(principal[np.arange(len(principal)), max_comp])
        signs[signs == 0] = 1
        principal = principal * signs[:, None]
    x = np.column_stack([
        np.log1p(inv["V"]), np.log1p(inv["surface"]),
        inv["compactness"], inv["elongation"],
        principal[:, 0], principal[:, 1], principal[:, 2],
        inv["chan"][:, 0], int_std,
    ]).astype(np.float32)
    for col in range(x.shape[1]):
        mu, sigma = float(x[:, col].mean()), float(x[:, col].std())
        if sigma > 1e-8: x[:, col] = (x[:, col] - mu) / sigma
        else: x[:, col] = 0.0
    if raw_ei.shape[1] > 0:
        ea = edge_invariants(raw_nf, raw_ei, raw_ef, C)
        ei_fwd = torch.tensor(raw_ei, dtype=torch.long)
        ei_rev = torch.stack([ei_fwd[1], ei_fwd[0]])
        edge_index = torch.cat([ei_fwd, ei_rev], dim=1)
        edge_attr = torch.tensor(np.concatenate([ea, ea]), dtype=torch.float32)
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 10), dtype=torch.float32)
    flat = labels_np.ravel(); valid = flat >= 0
    gt = (seg.ravel() == 2).astype(np.float64)
    max_id = int(flat[valid].max()) if valid.any() else -1
    y = np.zeros(n_sn, dtype=np.int64)
    if max_id >= 0:
        tc = np.bincount(flat[valid], weights=gt[valid], minlength=max_id + 1)
        total_c = np.bincount(flat[valid], minlength=max_id + 1)
        overlap = tc / np.maximum(total_c, 1)
        y[:min(n_sn, len(overlap))] = (overlap[:n_sn] >= overlap_th).astype(np.int64)
    return Data(x=torch.tensor(x, dtype=torch.float32),
                edge_index=edge_index, edge_attr=edge_attr,
                y=torch.tensor(y, dtype=torch.long))

pr("Helpers defined.")

In [ ]:
# ---- Discover volumes and split ----

all_vids = discover_volumes()
pr(f"Found {len(all_vids)} LiTS volumes with tumor")

np.random.seed(42)
perm = np.random.permutation(len(all_vids))
n_train = int(0.7 * len(all_vids)); n_val = int(0.15 * len(all_vids))
train_ids = sorted([all_vids[i] for i in perm[:n_train]])
val_ids = sorted([all_vids[i] for i in perm[n_train:n_train + n_val]])
test_ids = sorted([all_vids[i] for i in perm[n_train + n_val:]])
pr(f"Split: {len(train_ids)} train / {len(val_ids)} val / {len(test_ids)} test")

In [ ]:
# ---- Build / load Mode C graphs for all 118 volumes ----
# Reuse twostage cache if available, otherwise build from scratch

MAX_NODES_GPU = 500_000  # Skip volumes too large for single-graph GPU pass

graphs = {}; label_maps = {}; seg_maps = {}
t0_all = time.time()

for vid in train_ids + val_ids + test_ids:
    # Check our own cache first, then twostage cache
    cache_g = os.path.join(GRAPH_CACHE, f"graph_{vid}.pt")
    cache_l = os.path.join(GRAPH_CACHE, f"labels_{vid}.npy")
    cache_s = os.path.join(GRAPH_CACHE, f"seg_{vid}.npy")
    
    ts_g = os.path.join(TWOSTAGE_CACHE, f"graph_{vid}.pt")
    ts_l = os.path.join(TWOSTAGE_CACHE, f"labels_{vid}.npy")
    ts_s = os.path.join(TWOSTAGE_CACHE, f"seg_{vid}.npy")
    
    if os.path.exists(cache_g):
        g = torch.load(cache_g, weights_only=False)
        lnp = np.load(cache_l); s = np.load(cache_s)
    elif os.path.exists(ts_g):
        # Reuse twostage cache (same Mode C graphs)
        g = torch.load(ts_g, weights_only=False)
        lnp = np.load(ts_l); s = np.load(ts_s)
        # Copy to our cache for future runs
        torch.save(g, cache_g); np.save(cache_l, lnp); np.save(cache_s, s)
    else:
        # Build from scratch
        ct_raw, seg, ct_u8_full = load_and_convert(vid)
        organ_mask = (seg == 1) | (seg == 2)
        slc = bbox_from_mask(organ_mask, margin=32)
        ct_crop = ct_raw[slc]; seg_crop = seg[slc]; organ_crop = organ_mask[slc]
        ct_u8_crop = np.clip(ct_crop, HU_MIN, HU_MAX)
        ct_u8_crop = ((ct_u8_crop - HU_MIN) / (HU_MAX - HU_MIN) * 255).round().astype(np.uint8)
        ct_u8_crop = np.ascontiguousarray(ct_u8_crop[..., np.newaxis])
        protect = make_intensity_protected(ct_u8_crop, organ_crop)
        pm = np.ascontiguousarray(protect)
        raw_nf, raw_ei, raw_ef, raw_labels, raw_adj = fastloops.merge_and_cut_protected(
            ct_u8_crop, pm, merge_distance=PSI, cut_distance=ALPHA, connectivity="faces")
        lnp = np.asarray(raw_labels)
        s = seg_crop
        raw_nf = np.asarray(raw_nf); raw_ei = np.asarray(raw_ei); raw_ef = np.asarray(raw_ef)
        g = build_pyg_graph(raw_nf, raw_ei, raw_ef, lnp, s, ct_u8_crop, OVERLAP_TH)
        torch.save(g, cache_g); np.save(cache_l, lnp); np.save(cache_s, s)
    
    graphs[vid] = g; label_maps[vid] = lnp; seg_maps[vid] = s

dt_all = time.time() - t0_all
pr(f"Loaded {len(graphs)} graphs in {dt_all:.1f}s")

# Stats
sizes = [g.num_nodes for g in graphs.values()]
tu_counts = [int((g.y == 1).sum()) for g in graphs.values()]
pr(f"Node counts: min={min(sizes):,} median={int(np.median(sizes)):,} max={max(sizes):,} mean={int(np.mean(sizes)):,}")
pr(f"Tumor nodes: min={min(tu_counts):,} max={max(tu_counts):,} mean={int(np.mean(tu_counts)):,}")

## Step 2: IG-JEPA pre-training (self-supervised)

In [ ]:
# ---- GIN Encoder for JEPA ----

class GINEncoder(nn.Module):
    """GIN encoder with edge features via GINE convolutions."""
    def __init__(self, in_dim, edge_dim, hid, layers=3):
        super().__init__()
        self.proj = nn.Linear(in_dim, hid)
        self.edge_proj = nn.Linear(edge_dim, hid)
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(layers):
            mlp = nn.Sequential(nn.Linear(hid, hid), nn.ReLU(), nn.Linear(hid, hid))
            self.convs.append(GINEConv(mlp, edge_dim=hid))
            self.norms.append(nn.LayerNorm(hid))
        self.drop = nn.Dropout(0.1)

    def forward(self, x, edge_index, edge_attr=None):
        x = self.proj(x)
        if edge_attr is not None and edge_attr.numel() > 0:
            ea = self.edge_proj(edge_attr)
        else:
            ea = torch.zeros(edge_index.size(1), x.size(1), device=x.device)
        for conv, norm in zip(self.convs, self.norms):
            x = x + self.drop(F.gelu(norm(conv(x, edge_index, ea))))
        return x


class IGJEPA(nn.Module):
    """IG-JEPA: topology-aware masking + VICReg on graph supernodes.
    
    Uses GINE (edge-aware) encoder instead of plain GIN to leverage
    the rich edge features from the Rust crate.
    """
    def __init__(self, in_dim, edge_dim, hid=256, layers=3, mask_ratio=0.3, mom=0.996):
        super().__init__()
        self.mask_ratio = mask_ratio
        self.mom = mom
        self.hid = hid
        self.enc = GINEncoder(in_dim, edge_dim, hid, layers)
        self.tgt = copy.deepcopy(self.enc)
        for p in self.tgt.parameters():
            p.requires_grad = False
        self.pred = nn.Sequential(
            nn.Linear(hid, hid), nn.GELU(), nn.LayerNorm(hid),
            nn.Linear(hid, hid)
        )
        self.lam_var = 25.0
        self.lam_cov = 1.0

    @torch.no_grad()
    def ema_update(self):
        for p, t in zip(self.enc.parameters(), self.tgt.parameters()):
            t.data.mul_(self.mom).add_(p.data, alpha=1 - self.mom)

    def subgraph_mask(self, ei, N):
        """Random node masking."""
        nm = max(1, int(N * self.mask_ratio))
        perm = torch.randperm(N, device=ei.device)
        mi = perm[:nm].sort().values
        mask_set = torch.zeros(N, dtype=torch.bool, device=ei.device)
        mask_set[mi] = True
        ctx_ei = ei[:, ~mask_set[ei[0]] & ~mask_set[ei[1]]]
        inbound_ei = ei[:, ~mask_set[ei[0]] & mask_set[ei[1]]]
        return mi, ctx_ei, inbound_ei, mask_set

    def forward(self, x, edge_index, edge_attr=None):
        N = x.size(0)
        mi, ctx_ei, inbound_ei, mask_set = self.subgraph_mask(edge_index, N)

        # Teacher: full graph
        with torch.no_grad():
            tz = self.tgt(x, edge_index, edge_attr)
            tgt = F.layer_norm(tz[mi], [self.hid])

        # Student: context only
        xc = x.clone()
        xc[mi] = 0
        
        # Filter edge_attr for context edges
        if edge_attr is not None and edge_attr.numel() > 0:
            ctx_mask = ~mask_set[edge_index[0]] & ~mask_set[edge_index[1]]
            ctx_ea = edge_attr[ctx_mask]
        else:
            ctx_ea = None
        
        cz = self.enc(xc, ctx_ei, ctx_ea)

        # Aggregate context into masked positions via inbound edges
        if inbound_ei.size(1) > 0:
            src_emb = cz[inbound_ei[0]]
            dst_idx = inbound_ei[1]
            agg = torch.zeros(N, self.hid, device=x.device)
            counts = torch.zeros(N, 1, device=x.device)
            agg.index_add_(0, dst_idx, src_emb)
            counts.index_add_(0, dst_idx, torch.ones_like(dst_idx, dtype=torch.float32).unsqueeze(1))
            counts = counts.clamp(min=1)
            masked_ctx = agg[mi] / counts[mi]
        else:
            masked_ctx = torch.zeros(len(mi), self.hid, device=x.device)

        pred = F.layer_norm(self.pred(masked_ctx), [self.hid])
        loss_pred = F.smooth_l1_loss(pred, tgt)

        # VICReg on context embeddings
        ctx_emb = cz[~mask_set]
        Nc = ctx_emb.size(0)
        if Nc > 1:
            std = ctx_emb.std(dim=0)
            loss_var = F.relu(1.0 - std).mean()
            z_c = ctx_emb - ctx_emb.mean(dim=0)
            cov_mat = (z_c.T @ z_c) / max(Nc - 1, 1)
            loss_cov = (cov_mat.fill_diagonal_(0) ** 2).sum() / self.hid
        else:
            loss_var = torch.tensor(0.0, device=x.device)
            loss_cov = torch.tensor(0.0, device=x.device)
            std = torch.tensor(0.0)

        total = loss_pred + self.lam_var * loss_var + self.lam_cov * loss_cov
        return total, {"pred": loss_pred.item(), "var": loss_var.item(),
                       "cov": loss_cov.item(), "std": std.mean().item()}

    def encode(self, x, edge_index, edge_attr=None):
        return self.enc(x, edge_index, edge_attr)

pr("IG-JEPA model defined.")

In [ ]:
# ---- Pre-training ----

section("IG-JEPA Pre-training")

# Use trainable graphs (those that fit in GPU memory)
trainable = [v for v in train_ids if v in graphs and graphs[v].num_nodes <= MAX_NODES_GPU]
val_usable = [v for v in val_ids if v in graphs]
pr(f"Trainable: {len(trainable)}/{len(train_ids)} (max {MAX_NODES_GPU:,} nodes)")

# Model config
g0 = graphs[trainable[0]]
IN_DIM = g0.x.shape[1]  # 9
EDGE_DIM = g0.edge_attr.shape[1] if g0.edge_attr.numel() > 0 else 10
HID = 128
LAYERS = 3
PRETRAIN_EPOCHS = 50
MASK_RATIO = 0.3

pr(f"in_dim={IN_DIM}, edge_dim={EDGE_DIM}, hid={HID}, layers={LAYERS}")
pr(f"mask_ratio={MASK_RATIO}, pretrain_epochs={PRETRAIN_EPOCHS}")

model = IGJEPA(IN_DIM, EDGE_DIM, HID, LAYERS, mask_ratio=MASK_RATIO).to(DEVICE)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                        lr=3e-4, weight_decay=0.01)
sch = CosineAnnealingLR(opt, T_max=PRETRAIN_EPOCHS, eta_min=1e-6)

pretrain_history = []

for ep in range(1, PRETRAIN_EPOCHS + 1):
    model.train()
    total_loss = 0; total_pred = 0; total_var = 0; total_cov = 0
    processed = 0
    
    for vid in np.random.permutation(trainable):
        g = graphs[vid]
        try:
            gd = g.to(DEVICE)
            opt.zero_grad()
            loss, info = model(gd.x, gd.edge_index, gd.edge_attr)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            model.ema_update()
            total_loss += loss.item()
            total_pred += info["pred"]
            total_var += info["var"]
            total_cov += info["cov"]
            processed += 1
            del gd
        except torch.cuda.OutOfMemoryError:
            try: del gd
            except: pass
            torch.cuda.empty_cache()
            continue
        torch.cuda.empty_cache()
    
    sch.step()
    avg_loss = total_loss / max(processed, 1)
    avg_pred = total_pred / max(processed, 1)
    avg_var = total_var / max(processed, 1)
    avg_cov = total_cov / max(processed, 1)
    
    pretrain_history.append({"epoch": ep, "loss": avg_loss, "pred": avg_pred,
                             "var": avg_var, "cov": avg_cov, "processed": processed})
    
    if ep % 5 == 0 or ep <= 3:
        pr(f"  Ep {ep:3d} | Loss: {avg_loss:.4f} (pred={avg_pred:.4f} var={avg_var:.4f} cov={avg_cov:.4f}) | "
           f"std: {info.get('std', 0):.4f} | {processed} vols")

pr(f"\nPre-training complete. Final loss: {pretrain_history[-1]['loss']:.4f}")

## Step 3: Fine-tune with MLP probe

In [ ]:
section("Fine-tuning: JEPA encoder (frozen) + MLP probe")

# Freeze encoder
model.eval()
for p in model.enc.parameters():
    p.requires_grad = False

# Class weights
total_pos = sum(int((graphs[v].y == 1).sum()) for v in trainable)
total_neg = sum(int((graphs[v].y == 0).sum()) for v in trainable)
ratio = total_neg / max(total_pos, 1)
eff = min(np.sqrt(ratio), 30.0)
class_weight = torch.tensor([1.0, eff], dtype=torch.float32).to(DEVICE)
pr(f"Class weight: [1.0, {eff:.1f}] (ratio: {ratio:.0f}:1, pos={total_pos:,} neg={total_neg:,})")

# MLP probe
probe = nn.Sequential(
    nn.Linear(HID, HID), nn.GELU(), nn.Dropout(0.1),
    nn.Linear(HID, HID // 2), nn.GELU(),
    nn.Linear(HID // 2, 2),
).to(DEVICE)

probe_opt = torch.optim.Adam(probe.parameters(), lr=1e-3, weight_decay=1e-4)
PROBE_EPOCHS = 150
PATIENCE = 20
VAL_EVERY = 3

best_dice_frozen = -1.0
best_state_frozen = None
wait = 0
probe_history = []

for ep in range(1, PROBE_EPOCHS + 1):
    probe.train()
    total_loss = 0; processed = 0
    
    for vid in np.random.permutation(trainable):
        g = graphs[vid]
        try:
            gd = g.to(DEVICE)
            with torch.no_grad():
                emb = model.encode(gd.x, gd.edge_index, gd.edge_attr)
            logits = probe(emb)
            loss = F.cross_entropy(logits, gd.y, weight=class_weight)
            probe_opt.zero_grad()
            loss.backward()
            probe_opt.step()
            total_loss += loss.item(); processed += 1
            del gd, emb, logits, loss
        except torch.cuda.OutOfMemoryError:
            try: del gd
            except: pass
            torch.cuda.empty_cache(); continue
        torch.cuda.empty_cache()
    
    mean_loss = total_loss / max(processed, 1)
    
    if ep % VAL_EVERY == 0 or ep <= 3:
        probe.eval()
        tp = fp = fn = 0
        fg_rates = []
        with torch.no_grad():
            for vid in val_usable:
                g = graphs[vid]; lnp = label_maps[vid]; s = seg_maps[vid]
                try:
                    gd = g.to(DEVICE)
                    emb = model.encode(gd.x, gd.edge_index, gd.edge_attr)
                    preds = probe(emb).argmax(dim=1)
                    fg_rates.append(float((preds == 1).float().mean().cpu()))
                    preds = preds.cpu().numpy()
                    del gd, emb; torch.cuda.empty_cache()
                except (torch.cuda.OutOfMemoryError, RuntimeError):
                    try: del gd
                    except: pass
                    torch.cuda.empty_cache()
                    # CPU fallback
                    enc_cpu = model.enc.cpu(); probe_cpu = probe.cpu()
                    emb = enc_cpu(g.x, g.edge_index, g.edge_attr)
                    preds = probe_cpu(emb).argmax(dim=1).numpy()
                    fg_rates.append(float((torch.tensor(preds) == 1).float().mean()))
                    model.enc.to(DEVICE); probe.to(DEVICE)
                    del emb
                
                flat = lnp.ravel(); valid = flat >= 0
                if not valid.any(): continue
                mid = int(flat[valid].max())
                lut = np.zeros(mid + 1, dtype=np.int8)
                lut[:min(len(preds), mid + 1)] = preds[:min(len(preds), mid + 1)]
                pm = np.where(valid, lut[flat], 0).reshape(lnp.shape).astype(bool)
                gm = s == 2; inter = int((pm & gm).sum())
                tp += inter; fp += int(pm.sum()) - inter; fn += int(gm.sum()) - inter
        
        vd = 2 * tp / (2 * tp + fp + fn + 1e-8)
        mfg = np.mean(fg_rates) if fg_rates else 0.0
        probe_history.append({"epoch": ep, "loss": mean_loss, "val_dice": vd, "fg": mfg})
        
        if vd > best_dice_frozen:
            best_dice_frozen = vd
            best_state_frozen = {k: v.detach().cpu().clone() for k, v in probe.state_dict().items()}
            wait = 0; marker = " *"
        else:
            wait += 1; marker = ""
        
        pr(f"  Ep {ep:3d} | Loss: {mean_loss:.4f} | val_dice={vd:.4f} | fg={mfg:.4f} | ({processed} vols){marker}")
        
        if wait >= PATIENCE:
            pr(f"  Early stop at epoch {ep}, best val Dice (frozen) = {best_dice_frozen:.4f}")
            break
    else:
        if ep % 10 == 0:
            pr(f"  Ep {ep:3d} | Loss: {mean_loss:.4f} | ({processed} vols)")

pr(f"\nBest val Dice (frozen encoder + probe): {best_dice_frozen:.4f}")

In [ ]:
# ---- Fine-tune with PARTIALLY UNFROZEN encoder ----
# Unfreeze last 1 GIN layer to allow adaptation

section("Fine-tuning: JEPA encoder (partial unfreeze) + MLP probe")

# Unfreeze last conv layer + norm
for p in model.enc.convs[-1].parameters():
    p.requires_grad = True
for p in model.enc.norms[-1].parameters():
    p.requires_grad = True

# Reset probe
probe2 = nn.Sequential(
    nn.Linear(HID, HID), nn.GELU(), nn.Dropout(0.1),
    nn.Linear(HID, HID // 2), nn.GELU(),
    nn.Linear(HID // 2, 2),
).to(DEVICE)

# Separate LR: lower for encoder, higher for probe
ft_opt = torch.optim.Adam([
    {"params": [p for p in model.enc.parameters() if p.requires_grad], "lr": 1e-4},
    {"params": probe2.parameters(), "lr": 1e-3},
], weight_decay=1e-4)

FT_EPOCHS = 150
best_dice_ft = -1.0
best_state_ft_enc = None
best_state_ft_probe = None
wait = 0
ft_history = []

for ep in range(1, FT_EPOCHS + 1):
    model.enc.train()
    probe2.train()
    total_loss = 0; processed = 0
    
    for vid in np.random.permutation(trainable):
        g = graphs[vid]
        try:
            gd = g.to(DEVICE)
            emb = model.encode(gd.x, gd.edge_index, gd.edge_attr)
            logits = probe2(emb)
            loss = F.cross_entropy(logits, gd.y, weight=class_weight)
            ft_opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(list(model.enc.parameters()) + list(probe2.parameters()), 1.0)
            ft_opt.step()
            total_loss += loss.item(); processed += 1
            del gd, emb, logits, loss
        except torch.cuda.OutOfMemoryError:
            try: del gd
            except: pass
            torch.cuda.empty_cache(); continue
        torch.cuda.empty_cache()
    
    mean_loss = total_loss / max(processed, 1)
    
    if ep % VAL_EVERY == 0 or ep <= 3:
        model.enc.eval(); probe2.eval()
        tp = fp = fn = 0; fg_rates = []
        with torch.no_grad():
            for vid in val_usable:
                g = graphs[vid]; lnp = label_maps[vid]; s = seg_maps[vid]
                try:
                    gd = g.to(DEVICE)
                    emb = model.encode(gd.x, gd.edge_index, gd.edge_attr)
                    preds = probe2(emb).argmax(dim=1)
                    fg_rates.append(float((preds == 1).float().mean().cpu()))
                    preds = preds.cpu().numpy()
                    del gd, emb; torch.cuda.empty_cache()
                except (torch.cuda.OutOfMemoryError, RuntimeError):
                    try: del gd
                    except: pass
                    torch.cuda.empty_cache()
                    enc_cpu = model.enc.cpu(); probe_cpu = probe2.cpu()
                    emb = enc_cpu(g.x, g.edge_index, g.edge_attr)
                    preds = probe_cpu(emb).argmax(dim=1).numpy()
                    fg_rates.append(float((torch.tensor(preds) == 1).float().mean()))
                    model.enc.to(DEVICE); probe2.to(DEVICE)
                    del emb
                
                flat = lnp.ravel(); valid = flat >= 0
                if not valid.any(): continue
                mid = int(flat[valid].max())
                lut = np.zeros(mid + 1, dtype=np.int8)
                lut[:min(len(preds), mid + 1)] = preds[:min(len(preds), mid + 1)]
                pm = np.where(valid, lut[flat], 0).reshape(lnp.shape).astype(bool)
                gm = s == 2; inter = int((pm & gm).sum())
                tp += inter; fp += int(pm.sum()) - inter; fn += int(gm.sum()) - inter
        
        vd = 2 * tp / (2 * tp + fp + fn + 1e-8)
        mfg = np.mean(fg_rates) if fg_rates else 0.0
        ft_history.append({"epoch": ep, "loss": mean_loss, "val_dice": vd, "fg": mfg})
        
        if vd > best_dice_ft:
            best_dice_ft = vd
            best_state_ft_enc = {k: v.detach().cpu().clone() for k, v in model.enc.state_dict().items()}
            best_state_ft_probe = {k: v.detach().cpu().clone() for k, v in probe2.state_dict().items()}
            wait = 0; marker = " *"
        else:
            wait += 1; marker = ""
        
        pr(f"  Ep {ep:3d} | Loss: {mean_loss:.4f} | val_dice={vd:.4f} | fg={mfg:.4f} | ({processed} vols){marker}")
        
        if wait >= PATIENCE:
            pr(f"  Early stop at epoch {ep}, best val Dice (fine-tuned) = {best_dice_ft:.4f}")
            break
    else:
        if ep % 10 == 0:
            pr(f"  Ep {ep:3d} | Loss: {mean_loss:.4f} | ({processed} vols)")

pr(f"\nBest val Dice (partial fine-tune): {best_dice_ft:.4f}")

## Step 4: Vanilla GINE baseline (no pre-training)

In [ ]:
section("Vanilla GINE baseline (no pre-training)")

class GINE(nn.Module):
    def __init__(self, nd, ed, h=128):
        super().__init__()
        self.ep = nn.Linear(ed, h)
        def mlp(d): return nn.Sequential(nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Linear(h, h))
        self.c1 = GINEConv(mlp(nd), edge_dim=h); self.b1 = BatchNorm(h)
        self.c2 = GINEConv(mlp(h), edge_dim=h); self.b2 = BatchNorm(h)
        self.c3 = GINEConv(mlp(h), edge_dim=h); self.b3 = BatchNorm(h)
        self.head = nn.Linear(h, 2)
    def forward(self, x, ei, ea):
        if ea is not None and ea.numel() > 0: ea = self.ep(ea)
        else:
            n = x.size(0); ei = torch.stack([torch.arange(n, device=x.device)]*2)
            ea = torch.zeros(n, self.ep.out_features, device=x.device)
        x = F.relu(self.b1(self.c1(x, ei, ea)))
        x = F.relu(self.b2(self.c2(x, ei, ea)))
        x = F.relu(self.b3(self.c3(x, ei, ea)))
        return self.head(x)

gine_model = GINE(IN_DIM, EDGE_DIM).to(DEVICE)
gine_opt = torch.optim.Adam(gine_model.parameters(), lr=1e-3, weight_decay=1e-4)
GINE_EPOCHS = 200
GINE_PATIENCE = 15

best_dice_gine = -1.0
best_state_gine = None
wait = 0
gine_history = []

for ep in range(1, GINE_EPOCHS + 1):
    gine_model.train()
    total_loss = 0; processed = 0
    
    for vid in np.random.permutation(trainable):
        g = graphs[vid]
        try:
            gd = g.to(DEVICE)
            gine_opt.zero_grad()
            logits = gine_model(gd.x, gd.edge_index, gd.edge_attr)
            loss = F.cross_entropy(logits, gd.y, weight=class_weight)
            loss.backward(); gine_opt.step()
            total_loss += loss.item(); processed += 1
            del gd, logits, loss
        except torch.cuda.OutOfMemoryError:
            try: del gd
            except: pass
            torch.cuda.empty_cache(); continue
        torch.cuda.empty_cache()
    
    mean_loss = total_loss / max(processed, 1)
    
    if ep % VAL_EVERY == 0 or ep <= 3:
        gine_model.eval()
        tp = fp = fn = 0; fg_rates = []
        with torch.no_grad():
            for vid in val_usable:
                g = graphs[vid]; lnp = label_maps[vid]; s = seg_maps[vid]
                try:
                    gd = g.to(DEVICE)
                    preds = gine_model(gd.x, gd.edge_index, gd.edge_attr).argmax(dim=1).cpu().numpy()
                    fg_rates.append(float((torch.tensor(preds) == 1).float().mean()))
                    del gd; torch.cuda.empty_cache()
                except (torch.cuda.OutOfMemoryError, RuntimeError):
                    try: del gd
                    except: pass
                    torch.cuda.empty_cache()
                    mc = gine_model.cpu()
                    preds = mc(g.x, g.edge_index, g.edge_attr).argmax(dim=1).numpy()
                    fg_rates.append(float((torch.tensor(preds) == 1).float().mean()))
                    gine_model.to(DEVICE)
                
                flat = lnp.ravel(); valid = flat >= 0
                if not valid.any(): continue
                mid = int(flat[valid].max())
                lut = np.zeros(mid + 1, dtype=np.int8)
                lut[:min(len(preds), mid + 1)] = preds[:min(len(preds), mid + 1)]
                pm = np.where(valid, lut[flat], 0).reshape(lnp.shape).astype(bool)
                gm = s == 2; inter = int((pm & gm).sum())
                tp += inter; fp += int(pm.sum()) - inter; fn += int(gm.sum()) - inter
        
        vd = 2 * tp / (2 * tp + fp + fn + 1e-8)
        mfg = np.mean(fg_rates) if fg_rates else 0.0
        gine_history.append({"epoch": ep, "loss": mean_loss, "val_dice": vd, "fg": mfg})
        
        if vd > best_dice_gine:
            best_dice_gine = vd
            best_state_gine = {k: v.detach().cpu().clone() for k, v in gine_model.state_dict().items()}
            wait = 0; marker = " *"
        else:
            wait += 1; marker = ""
        
        pr(f"  Ep {ep:3d} | Loss: {mean_loss:.4f} | val_dice={vd:.4f} | fg={mfg:.4f} | ({processed} vols){marker}")
        
        if wait >= GINE_PATIENCE:
            pr(f"  Early stop at epoch {ep}, best val Dice (GINE) = {best_dice_gine:.4f}")
            break
    else:
        if ep % 10 == 0:
            pr(f"  Ep {ep:3d} | Loss: {mean_loss:.4f} | ({processed} vols)")

if best_state_gine: gine_model.load_state_dict(best_state_gine)
pr(f"\nBest val Dice (vanilla GINE): {best_dice_gine:.4f}")

## Step 5: Final evaluation on all splits

In [ ]:
section("Final Evaluation")

def evaluate_model(name, encode_fn, all_vids_eval, graphs, label_maps, seg_maps, train_ids, val_ids, test_ids):
    """Evaluate a model on all splits, returning per-volume results."""
    results = []
    for vid in sorted(all_vids_eval):
        if vid not in graphs: continue
        g = graphs[vid]; lnp = label_maps[vid]; s = seg_maps[vid]
        with torch.no_grad():
            try:
                preds = encode_fn(g, DEVICE)
            except:
                torch.cuda.empty_cache()
                preds = encode_fn(g, "cpu")
        
        flat = lnp.ravel(); valid = flat >= 0
        pm = np.zeros(lnp.shape, dtype=bool)
        if valid.any():
            mid = int(flat[valid].max())
            lut = np.zeros(mid + 1, dtype=np.int8)
            lut[:min(len(preds), mid + 1)] = preds[:min(len(preds), mid + 1)]
            pm = np.where(valid, lut[flat], 0).reshape(lnp.shape).astype(bool)
        gm = s == 2; inter = int((gm & pm).sum())
        dice = 2.0 * inter / (gm.sum() + pm.sum() + 1e-8)
        rec = inter / (gm.sum() + 1e-8)
        prec = inter / (pm.sum() + 1e-8) if pm.sum() > 0 else 0.0
        split = "train" if vid in train_ids else ("val" if vid in val_ids else "test")
        results.append({"vid": vid, "split": split, "dice": float(dice),
                        "recall": float(rec), "precision": float(prec)})
    return results


# --- Evaluate JEPA frozen probe ---
if best_state_frozen is not None:
    probe.load_state_dict(best_state_frozen)
    probe.eval()
    # Re-freeze encoder fully for evaluation
    for p in model.enc.parameters(): p.requires_grad = False
    model.enc.eval()

def jepa_frozen_fn(g, dev):
    gd = g.to(dev)
    emb = model.enc.to(dev)(gd.x, gd.edge_index, gd.edge_attr)
    p = probe.to(dev)(emb).argmax(dim=1).cpu().numpy()
    model.enc.to(DEVICE); probe.to(DEVICE)
    del gd, emb; torch.cuda.empty_cache()
    return p

all_eval_vids = train_ids + val_ids + test_ids
results_frozen = evaluate_model("JEPA frozen", jepa_frozen_fn, all_eval_vids,
                                graphs, label_maps, seg_maps, train_ids, val_ids, test_ids)

# --- Evaluate JEPA fine-tuned ---
if best_state_ft_enc is not None:
    model.enc.load_state_dict(best_state_ft_enc)
if best_state_ft_probe is not None:
    probe2.load_state_dict(best_state_ft_probe)
model.enc.eval(); probe2.eval()

def jepa_ft_fn(g, dev):
    gd = g.to(dev)
    emb = model.enc.to(dev)(gd.x, gd.edge_index, gd.edge_attr)
    p = probe2.to(dev)(emb).argmax(dim=1).cpu().numpy()
    model.enc.to(DEVICE); probe2.to(DEVICE)
    del gd, emb; torch.cuda.empty_cache()
    return p

results_ft = evaluate_model("JEPA fine-tuned", jepa_ft_fn, all_eval_vids,
                            graphs, label_maps, seg_maps, train_ids, val_ids, test_ids)

# --- Evaluate vanilla GINE ---
gine_model.eval()

def gine_fn(g, dev):
    gd = g.to(dev)
    p = gine_model.to(dev)(gd.x, gd.edge_index, gd.edge_attr).argmax(dim=1).cpu().numpy()
    gine_model.to(DEVICE)
    del gd; torch.cuda.empty_cache()
    return p

results_gine = evaluate_model("Vanilla GINE", gine_fn, all_eval_vids,
                              graphs, label_maps, seg_maps, train_ids, val_ids, test_ids)

pr("Evaluation complete.")

In [ ]:
# ---- Summary table ----

section("COMPARISON SUMMARY")

pr(f"{'Method':<30s} {'train Dice':>12s} {'val Dice':>12s} {'test Dice':>12s}")
pr("-" * 70)

for name, results in [("JEPA frozen probe", results_frozen),
                       ("JEPA partial fine-tune", results_ft),
                       ("Vanilla GINE (no pretrain)", results_gine)]:
    for split in ["train", "val", "test"]:
        scores = [r["dice"] for r in results if r["split"] == split]
        if not scores: scores = [0.0]
        if split == "train":
            line = f"  {name:<28s} {np.mean(scores):12.4f}"
        elif split == "val":
            line += f" {np.mean(scores):12.4f}"
        else:
            line += f" {np.mean(scores):12.4f}"
            pr(line)

pr(f"\n  Paper target: LiTS tumor Dice = 0.891 +/- 0.007")
pr(f"  Previous vanilla GINE (twostage notebook): val Dice = 0.32")
pr(f"")
pr(f"  Best JEPA frozen probe:     val Dice = {best_dice_frozen:.4f}")
pr(f"  Best JEPA partial fine-tune: val Dice = {best_dice_ft:.4f}")
pr(f"  Best vanilla GINE:          val Dice = {best_dice_gine:.4f}")

# Detailed per-split stats
pr(f"\n--- Detailed per-split (val) ---")
for name, results in [("JEPA frozen", results_frozen), ("JEPA fine-tune", results_ft), ("GINE", results_gine)]:
    val_scores = [r for r in results if r["split"] == "val"]
    if val_scores:
        dices = [r["dice"] for r in val_scores]
        recs = [r["recall"] for r in val_scores]
        precs = [r["precision"] for r in val_scores]
        pr(f"  {name:<20s}: Dice={np.mean(dices):.4f}+/-{np.std(dices):.4f}  "
           f"Recall={np.mean(recs):.4f}  Prec={np.mean(precs):.4f}")

In [ ]:
# ---- Save everything ----

torch.save({
    "jepa_model": model.state_dict(),
    "probe_frozen": best_state_frozen,
    "probe_ft": best_state_ft_probe,
    "enc_ft": best_state_ft_enc,
    "gine": best_state_gine,
}, os.path.join(RESULTS_DIR, "checkpoint.pt"))

summary = {
    "params": {
        "psi": PSI, "alpha": ALPHA, "hu": [HU_MIN, HU_MAX],
        "overlap_th": OVERLAP_TH, "hid": HID, "layers": LAYERS,
        "in_dim": IN_DIM, "edge_dim": EDGE_DIM,
        "mask_ratio": MASK_RATIO, "pretrain_epochs": PRETRAIN_EPOCHS,
    },
    "best_val_dice_frozen": float(best_dice_frozen),
    "best_val_dice_ft": float(best_dice_ft),
    "best_val_dice_gine": float(best_dice_gine),
    "pretrain_history": pretrain_history,
    "probe_history": probe_history,
    "ft_history": ft_history,
    "gine_history": gine_history,
    "results_frozen": results_frozen,
    "results_ft": results_ft,
    "results_gine": results_gine,
}
with open(os.path.join(RESULTS_DIR, "results.json"), "w") as f:
    json.dump(summary, f, indent=2)

pr(f"\nSaved to {RESULTS_DIR}/")
pr("Done.")